<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Krabby Real Estate Formula
</font>
</h1>

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Dataset
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
In this section, <code>import</code> your required libraries and tools, and read the data files saved in the <code>Data</code> folder under the names <code>train.csv</code> and <code>test.csv</code>, and load them into your workspace.
</font>
</p>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Train price NaN: {train_df["price"].isna().sum()}')
train_df.head()

<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Preprocessing and Feature Engineering
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;In this question, you can use any preprocessing/feature engineering technique of your choice.
&nbsp;&nbsp;&nbsp;&nbsp;<br>
&nbsp;&nbsp;&nbsp;&nbsp;The techniques you use will <b>not</b> be directly evaluated by the judging system. Rather, they will all impact the accuracy of your model; therefore, the better preprocessing/feature engineering you perform to improve the model's accuracy, the more points you will earn from this question.
&nbsp;&nbsp;&nbsp;&nbsp;In this section, you can allocate a portion of the available data for validation.
</font>
</p>

In [ ]:
train_df = train_df.dropna(subset=['price'])
y = train_df['price'].values.copy()

feature_cols = [c for c in train_df.columns if c != 'price']
n_train = len(train_df)
n_test = len(test_df)

train_features = train_df[feature_cols].copy()
test_features = test_df[feature_cols].copy()
all_features = pd.concat([train_features, test_features], ignore_index=True)
print(f'All features: {all_features.shape}')

for col in all_features.columns:
    if all_features[col].dtype in ['float64', 'int64', 'float32', 'int32']:
        all_features[col] = all_features[col].fillna(all_features[col].median())
    else:
        mv = all_features[col].mode()
        if len(mv) > 0:
            all_features[col] = all_features[col].fillna(mv[0])

all_features['area_per_room'] = all_features['area'] / (all_features['rooms'] + 1)
all_features['floor_ratio'] = all_features['floor_number'] / (all_features['total_floors'] + 1)
all_features['age_per_floor'] = all_features['age'] / (all_features['total_floors'] + 1)
all_features['levy_per_area'] = all_features['monthly_levy'] / (all_features['area'] + 1)
all_features['area_squared'] = all_features['area'] ** 2
all_features['rooms_area'] = all_features['rooms'] * all_features['area']
all_features['total_floors_squared'] = all_features['total_floors'] ** 2
all_features['age_squared'] = all_features['age'] ** 2

cat_cols = all_features.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    all_features[col] = le.fit_transform(all_features[col].astype(str))

all_features = all_features.fillna(0)

X = all_features.iloc[:n_train]
X_test = all_features.iloc[n_train:]
print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Training Model
</font>
</h2>

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

xgb_model = XGBRegressor(n_estimators=2000, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0)
lgbm_model = LGBMRegressor(n_estimators=2000, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
gb_model = GradientBoostingRegressor(n_estimators=1000, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42)
rf_model = RandomForestRegressor(n_estimators=1000, max_depth=15, random_state=42, n_jobs=-1)

xgb_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

for name, model in [('XGB', xgb_model), ('LGBM', lgbm_model), ('GB', gb_model), ('RF', rf_model)]:
    val_pred = model.predict(X_val)
    val_r2 = r2_score(y_val, val_pred)
    print(f'{name} Val R2: {val_r2:.6f}')

xgb_model.fit(X, y)
lgbm_model.fit(X, y)
gb_model.fit(X, y)
rf_model.fit(X, y)
print('All models retrained on full data.')

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Evaluation Metric
</font>
</h2>

In [ ]:
from sklearn.metrics import r2_score

xgb_vp = xgb_model.predict(X_val)
lgbm_vp = lgbm_model.predict(X_val)
gb_vp = gb_model.predict(X_val)
rf_vp = rf_model.predict(X_val)
ensemble_vp = 0.35 * xgb_vp + 0.35 * lgbm_vp + 0.20 * gb_vp + 0.10 * rf_vp
val_r2 = r2_score(y_val, ensemble_vp)
score = max(0, 100 * (1 - (1 - val_r2) / 0.04))
print(f'Ensemble Val R2: {val_r2:.6f}')
print(f'Competition Score: {score:.2f}')

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Prediction on Test Dataset
</font>
</h2>

In [ ]:
xgb_tp = xgb_model.predict(X_test)
lgbm_tp = lgbm_model.predict(X_test)
gb_tp = gb_model.predict(X_test)
rf_tp = rf_model.predict(X_test)
ensemble_tp = 0.35 * xgb_tp + 0.35 * lgbm_tp + 0.20 * gb_tp + 0.10 * rf_tp

submission = pd.DataFrame({'price': ensemble_tp})
submission.to_csv('submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
submission.head()

<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

In [ ]:
import zipfile
import os

if not os.path.exists(os.path.join(os.getcwd(), 'house_prediction.ipynb')):
    %notebook -e house_prediction.ipynb

def compress(file_names):
    print('File Paths:')
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile('result.zip', mode='w') as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['house_prediction.ipynb', 'submission.csv']
compress(file_names)